[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kauelopes/topicos-ml/blob/main/aulas/02-python-numpy-jax/notebooks/02_lab.ipynb)

# Aula 2 — Laboratório: inflação acumulada em 12 meses

Vamos trabalhar com a variação mensal do **IPCA** (série 433 do SGS/Banco Central).

**Tarefas**
1. Carregar a série e inspecionar o array
2. Calcular a inflação acumulada em 12 meses **com um laço**
3. Fazer o mesmo **sem laços** (NumPy)
4. Classificar cada mês em relação à meta (3% ± 1,5 p.p.)
5. JAX: derivar a inflação acumulada em relação a uma taxa mensal
6. *Extensão*: gráfico

> Substitua cada `...` por código. Execute as células na ordem.

## 1. Carregar os dados

In [ ]:
import numpy as np
import pandas as pd

URL = ("https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados"
       "?formato=json&dataInicial=01/01/2015")

try:
    df = pd.read_json(URL)
    datas = pd.to_datetime(df["data"], dayfirst=True).to_numpy()
    pi = df["valor"].astype(float).to_numpy()
    print(f"IPCA carregado do BCB: {len(pi)} meses")
except Exception as erro:  # sem internet: série sintética para não travar a aula
    print("Falha ao acessar o BCB, usando dados simulados:", erro)
    rng = np.random.default_rng(1)
    datas = pd.date_range("2015-01-01", periods=120, freq="MS").to_numpy()
    pi = np.round(0.35 + 0.25 * rng.standard_normal(120), 2)

pi[:12]

In [ ]:
# TODO: veja a forma, o tipo e os valores mínimo e máximo de `pi`
...

## 2. Acumulado em 12 meses com laço
Para cada mês $t \geq 12$: $\pi^{12m}_t = 100\left[\prod_{s=t-11}^{t}(1+\pi_s/100) - 1\right]$

In [ ]:
def acum_12m_laco(pi):
    saida = []
    for t in range(11, len(pi)):
        fator = 1.0
        # TODO: acumule (1 + pi[s]/100) para os 12 meses que terminam em t
        ...
        saida.append(100 * (fator - 1))
    return np.array(saida)

a_laco = acum_12m_laco(pi)
a_laco[-5:]

## 3. Agora sem laços
Dica: construa o número-índice $I_t = I_0\prod_{s\le t}(1+\pi_s/100)$ com `np.cumprod`. Então $\pi^{12m}_t = 100\,(I_t/I_{t-12} - 1)$.

In [ ]:
def acum_12m_vet(pi):
    indice = ...   # TODO: número-índice com I_0 = 100 (tamanho len(pi) + 1)
    return ...     # TODO: acumulado em 12 meses sem laços

a_vet = acum_12m_vet(pi)
np.allclose(a_laco, a_vet)

In [ ]:
# TODO: compare os tempos das duas versões com %timeit
...

## 4. Classificar em relação à meta (3% ± 1,5 p.p.)

> Simplificação: a meta foi de 4,5% até 2018 e caiu gradualmente até 3% (2024). Aqui usamos 3% para todo o período; como extensão, use a meta de cada ano.

In [ ]:
centro, tol = 3.0, 1.5
acima = ...    # TODO: array booleano
abaixo = ...
dentro = ...
# TODO: quantos meses em cada situação? (dica: .sum() em booleanos conta os True)
...

## 5. JAX: sensibilidade do acumulado
Quanto o acumulado dos **últimos 12 meses** muda se a inflação de cada mês subir 1 p.p.?
Isto é, calcule $\partial \pi^{12m} / \partial \pi_s$ para cada um dos 12 meses com `jax.grad`.
Compare com a expressão analítica $\prod_{r\neq s}(1+\pi_r/100)$.

In [ ]:
import jax
import jax.numpy as jnp

def acum(p):
    return ...   # TODO: acumulado (%) de um vetor de taxas mensais, usando jnp

ult12 = jnp.array(pi[-12:])
sens = ...       # TODO: gradiente de `acum` em `ult12`
analitica = ...  # TODO
sens, jnp.allclose(sens, analitica)

**Pergunta:** por que todas as sensibilidades são próximas de 1? Em que situação elas seriam bem diferentes entre si?

## 6. Extensão: gráfico

In [ ]:
import matplotlib.pyplot as plt
# TODO: plote a_vet contra datas[11:] e sombreie o intervalo da meta (ax.axhspan)
...